<a href="https://colab.research.google.com/github/Ridzzz0Alam/data_cleaning_and_preprocessing/blob/main/Assignment_04__Comprehensive_EDA_%26_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Professional Data Cleaning Notebook
## Problem Detection → Interpretation → Solution → Code

This notebook serves as a practical interview cheat sheet and project template for Data Analysts, Data Scientists, and ML Engineers.


In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("Garment_Job_Risk_bulk_dataset.csv")
df.head(10)
df.tail(10)


,Name,Garment_Name,Experience_Years,Job_Role,Working_Hours_Per_Week,Exposure_To_Chemicals,Machine_Use,Workstation_Ergonomics,Injury_History,Health_Condition,Safety_Training,Shift_Type,Task_Rotation_Frequency,Job_Risk
2490,Rasel Hossain,Pallmall Group,3,Dyeing,52,Medium,NaN,Average,0,Healthy,Basic,Day,Occasional,Medium
2491,Mariam Noor,Pallmall Group,23,Sewing,40,Low,Light,Poor,1,Healthy,Basic,Day,Frequent,Medium
2492,Abdul Latif,Pallmall Group,33,Cutting,48,NaN,Light,Poor,0,Healthy,Basic,Day,Rare,Medium
2493,Fariha Khan,Pallmall Group,29,Packaging,58,Medium,Heavy,Good,0,Healthy,Advanced,Day,Frequent,Medium
2494,Fariha Khan,Pallmall Group,17,Cutting,64,Medium,Light,Poor,1,Healthy,NaN,Evening,Frequent,High
2495,Shamina Begum,Pallmall Group,18,Dyeing,32,NaN,Heavy,Average,0,Healthy,Basic,Day,Occasional,Medium
2496,Fariha Khan,Pallmall Group,30,Dyeing,40,Medium,Light,Average,1,Healthy,Basic,Day,Frequent,Medium
2497,Fariha Khan,Pallmall Group,18,Packaging,55,High,Light,Poor,2,Healthy,NaN,Night,Occasional,High
2498,Mushfiqur Rahman,Pallmall Group,18,Cutting,52,High,Light,Average,0,Minor_Chronic,Basic,Day,Occasional,Low
2499,Mushfiqur Rahman,Pallmall Group,21,Packaging,62,Medium,NaN,Average,1,Minor_Chronic,Basic,Day,Occasional,Low


## 1. Missing Values Handling

### Problem Detection

```python
df.isnull().sum()
df.isnull().mean()*100
```

### Interpretation

- Few missing values (<5%) → usually safe to drop.
- Numeric columns → Mean/Median.
- Categorical columns → Mode.
- Time series → Forward Fill / Backward Fill.

### Solutions

- Drop rows
- Drop columns
- Mean Imputation
- Median Imputation
- Mode Imputation
- Forward Fill
- Backward Fill


In [13]:
df.isnull().sum()
df.isnull().mean()*100

# Drop rows
#df.dropna(inplace=True)

# Mean
#df["salary"] = df["salary"].fillna(df["salary"].mean())

# Median
#df["age"] = df["age"].fillna(df["age"].median())

# Mode
#df["city"] = df["city"].fillna(df["city"].mode()[0])

# Forward Fill
#df["sales"] = df["sales"].ffill()

# Backward Fill
#df["sales"] = df["sales"].bfill()


"""
There are so many datasets missing in these columns (Exposure_To_Chemicals, Machine_Use, Safety_Training)
so for that I have decided to use Mode imputation to fill up the missing values as Mean/Median is for numeric columns.
Forward/Backward is used for time series data and dropping the datas of huge number will change the meaning of the data
itself.
"""

df["Exposure_To_Chemicals"] = df["Exposure_To_Chemicals"].fillna(df["Exposure_To_Chemicals"].mode()[0])
df["Machine_Use"] = df["Machine_Use"].fillna(df["Machine_Use"].mode()[0])
df["Safety_Training"] = df["Safety_Training"].fillna(df["Safety_Training"].mode()[0])



## 2. Duplicate Data Removal

### Problem Detection

```python
df.duplicated().sum()
```

### Interpretation

- Duplicate records create bias.
- Same customer appearing multiple times can distort analysis.

### Solutions

- Remove exact duplicates.
- Remove duplicate IDs.
- Keep latest record only.


In [14]:
df.duplicated().sum()

# Exact duplicates
df.drop_duplicates(inplace=True)

"""
But after seeing the table,There is no unique ID column in this dataset,
so I have decided to not use this step

df.drop_duplicates(
    subset=["customer_id"],
    keep="first"
)
"""



'\nBut after seeing the table,There is no unique ID column in this dataset,\nso I have decided to not use this step\n\ndf.drop_duplicates(\n    subset=["customer_id"],\n    keep="first"\n)\n'

## 3. Wrong Format Fix

### Problem Detection

```python
df.dtypes
```

Common Issues:

- Dates stored as object
- Numbers stored as strings
- Currency/unit symbols inside values

### Solutions

- Convert date formats
- Convert strings to numeric
- Normalize categorical values


In [15]:
df.dtypes

"""
SInce the dataset has no date column and no currency/ unit symbols inside values,
the other operations are not used and commented out
"""

# Date conversion
# df["date"] = pd.to_datetime(
#     df["date"],
#     errors="coerce"
# )

# Currency cleanup
# df["price"] = (
#     df["price"]
#     .astype(str)
#     .str.replace("$","",regex=False)
# )

# df["price"] = pd.to_numeric(
#     df["price"],
#     errors="coerce"
# )

"""
Category normalization fits the best in this dataset as the column
Garment_Name has single consistent value already.
"""
df["Garment_Name"] = df["Garment_Name"].replace({
    "Pallmall Group":"Pallmall Group"
})


## 4. Inconsistent Labels

### Problem Detection

```python
df["city"].unique()
df["brand"].unique()
```

### Interpretation

Same entity represented differently.

Example:

- Apple
- apple
- APPLE

### Solutions

- Case normalization
- Spelling correction
- Synonym mapping


In [16]:
df["Job_Role"].unique()
df["Job_Risk"].unique()

# All categorical columns in this dataset (Job_Role, Exposure_To_Chemicals,
# Machine_Use, Workstation_Ergonomics, Health_Condition, Safety_Training,
# Shift_Type, Task_Rotation_Frequency, Job_Risk) already use one
# consistent spelling/casing scheme, so no spelling correction or
# synonym mapping is needed. Case normalization is applied defensively.

"""
All categorical columns on the dataset already use one consistent spelling method, so no
spelling correction or synonym is needed. LowerCase is applied
"""

# Lower Case
df["Job_Role"] = df["Job_Role"].str.lower()

# Spelling correction
# df["city"] = df["city"].replace({
#     "Dhka":"Dhaka"
# })

# Synonyms
# df["vehicle"] = df["vehicle"].replace({
#     "automobile":"car"
# })


## 5. Outlier Detection & Handling

### Problem Detection

Methods:

- Boxplot
- IQR
- Z-Score
- Domain Rules

Example:

Age < 0
Salary > 10M

### Solutions

- Remove
- Cap (Winsorization)
- Log Transform


In [17]:
Q1 = df["Experience_Years"].quantile(0.25)
Q3 = df["Experience_Years"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

"""
Experience_Years (0-40 yrs) and Working_Hours_Per_Week (28-68 hrs/wk)
fall within the range, so no rows are removed/capped here.The Remove/Cap/Log
Transform patterns are kept commented.
"""

# Detect
outliers = df[
    (df["Experience_Years"] < lower) |
    (df["Experience_Years"] > upper)
]
outliers

# Experience_Years (0-40 yrs) and Working_Hours_Per_Week (28-68 hrs/wk)
# fall within plausible domain ranges, so no rows are removed/capped here.
# The Remove / Cap / Log Transform patterns are kept as reference.

# Remove
# df = df[
#     (df["salary"] >= lower) &
#     (df["salary"] <= upper)
# ]

# Cap
# df["salary"] = df["salary"].clip(
#     lower,
#     upper
# )

# Log Transform
# df["salary"] = np.log1p(df["salary"])


,Name,Garment_Name,Experience_Years,Job_Role,Working_Hours_Per_Week,Exposure_To_Chemicals,Machine_Use,Workstation_Ergonomics,Injury_History,Health_Condition,Safety_Training,Shift_Type,Task_Rotation_Frequency,Job_Risk


## 6. Noisy Data Filtering

### Problem Detection

Examples:

- Invalid ages
- Typing mistakes
- Random sensor noise
- Blurry images
- Irrelevant text samples

### Solutions

- Rule-based filtering
- Threshold filtering
- Quality filtering


In [18]:
# Invalid experience
df = df[df["Experience_Years"] >= 0]

# Remove short reviews
# df = df[df["review"].str.len() > 5]

# Image example
# if blur_score < 100:
#     discard_image()


## 7. Data Type Consistency Check

### Problem Detection

```python
df.dtypes
```

Common Problems:

- int stored as object
- boolean stored as string
- category stored as object

### Solutions

Convert to correct data types.


In [19]:
df["Experience_Years"] = df["Experience_Years"].astype(int)
df["Working_Hours_Per_Week"] = df["Working_Hours_Per_Week"].astype(int)
df["Injury_History"] = df["Injury_History"].astype(int)

# No boolean as a string column exists in this dataset so kept commented
# df["active"] = df["active"].map({
#     "True":True,
#     "False":False
# })

df["Job_Role"] = df["Job_Role"].astype("category")
df["Job_Risk"] = df["Job_Risk"].astype("category")


## 8. Schema Validation

### Problem Detection

Check:

- Missing columns
- Extra columns
- Wrong column names

### Solutions

Validate schema before pipeline execution.


In [20]:
required_cols = [
    "Name",
    "Garment_Name",
    "Experience_Years",
    "Job_Role",
    "Working_Hours_Per_Week",
    "Exposure_To_Chemicals",
    "Machine_Use",
    "Workstation_Ergonomics",
    "Injury_History",
    "Health_Condition",
    "Safety_Training",
    "Shift_Type",
    "Task_Rotation_Frequency",
    "Job_Risk"
]

missing = (
    set(required_cols)
    - set(df.columns)
)

extra = (
    set(df.columns)
    - set(required_cols)
)

print("Missing:", missing)
print("Extra:", extra)


Missing: set()
Extra: set()


## 9. Data Range Validation

### Problem Detection

Examples:

- age > 150
- price < 0
- probability > 1

### Solutions

Apply business/domain rules.


In [21]:
# Experience_Years
df = df[
    (df["Experience_Years"] >= 0) &
    (df["Experience_Years"] <= 50)
]

# Working_Hours_Per_Week
df = df[df["Working_Hours_Per_Week"] >= 0]

# Injury_History ranges from 1 to 6 but I have added the range 10 just in case
df = df[
    (df["Injury_History"] >= 0) &
    (df["Injury_History"] <= 10)
]


## 10. Imbalanced Data Check

### Problem Detection

```python
df["target"].value_counts()
df["target"].value_counts(normalize=True)*100
```

Example:

Class 0 → 95%
Class 1 → 5%

Interpretation:

Highly Imbalanced Dataset

### Solutions

1. Oversampling
2. Undersampling
3. SMOTE
4. Class Weighting
5. Better Evaluation Metrics


In [22]:
# Check distribution
print(df["Job_Risk"].value_counts())
print(df["Job_Risk"].value_counts(normalize=True)*100)

"""
Job_Risk is Low ~42% / Medium ~36% / High ~22% — a mild, not severe,
imbalance (no class below 10%). Class Weighting and Better Evaluation
Metrics is the recommended approach here. The heavier resampling
techniques such as Oversampling/Undersampling/SMOTE are kept as comment
since this dataset does not need them.
"""


# ---------------------
# Oversampling
# ---------------------
# from sklearn.utils import resample

# majority = df[df.Job_Risk == "Low"]
# minority = df[df.Job_Risk == "High"]
#
# minority_up = resample(
#     minority,
#     replace=True,
#     n_samples=len(majority),
#     random_state=42
# )

# df_balanced = pd.concat(
#     [majority, minority_up]
# )

# ---------------------
# Undersampling
# ---------------------

# majority_down = resample(
#     majority,
#     replace=False,
#     n_samples=len(minority),
#     random_state=42
# )
#
# df_under = pd.concat(
#     [majority_down, minority]
# )

# ---------------------
# SMOTE
# ---------------------

# from imblearn.over_sampling import SMOTE
#
# X = df.drop("Job_Risk", axis=1)
# y = df["Job_Risk"]
#
# smote = SMOTE(random_state=42)

# X_resampled, y_resampled = (
#     smote.fit_resample(X, y)
# )

# ---------------------
# Class Weight
# ---------------------

# LogisticRegression(
#     class_weight="balanced"
# )

# ---------------------
# Better Metrics
# ---------------------

# precision_score
# recall_score
# f1_score
# roc_auc_score

# Saving the cleaned dataset
df.to_csv("Cleaned_Garment_Job_Risk_bulk_dataset.csv", index=False)
print("Cleaned dataset saved. Final shape:", df.shape)


Job_Risk
Low       1050
Medium     900
High       550
Name: count, dtype: int64
Job_Risk
Low       42.0
Medium    36.0
High      22.0
Name: proportion, dtype: float64
Cleaned dataset saved. Final shape: (2500, 14)
